# Data Cleaning and SQLite Validation

This notebook completes the remaining data-foundation assignment tasks: remove duplicates, standardize inconsistent categories, handle domain outliers, write cleaned data to SQLite, and validate results independently with Pandas and SQL.

The generated files under `data/raw/` remain unchanged.

In [1]:
from pathlib import Path
import sqlite3
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "generate_data.py").is_file():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.data_pipeline import cleaning_metrics, run_pipeline

result = run_pipeline(
    raw_dir=ROOT / "data" / "raw",
    processed_dir=ROOT / "data" / "processed",
    db_path=ROOT / "data" / "instamart.db",
    output_dir=ROOT / "outputs",
)
raw = result["raw"]
clean = result["clean"]
print("Validation checkpoints:", result["checkpoints"])

Validation checkpoints: {'customers': 1200, 'products': 40, 'orders': 12309, 'tickets': 164}


## 1. Cleaning rules and before/after counts

Each change uses an explicit, repeatable rule. Blank and overlong ticket text is intentionally retained for later guardrail handling rather than inventing customer messages.

In [2]:
metrics = cleaning_metrics(raw, clean)
display(metrics)

,dataset,metric,before,after,cleaning_rule
0,customers,rows,1200,1200,preserved
1,customers,city variants,24,6,strip whitespace and title-case
2,customers,gender values including blanks,11,3,"map to Male, Female, or Unknown"
3,customers,invalid or blank ages,67,0,replace with city median
4,orders,rows,13084,12309,keep latest final state
5,orders,exact duplicate rows,119,0,remove exact duplicates
6,orders,repeated order ID excess,775,0,keep latest timestamp per order
7,orders,pending rows,656,0,resolve to later final state
8,orders,nonnumeric amounts,120,0,"recalculate from product price, quantity, and ..."
9,orders,negative delivery durations,106,0,replace and impute completed orders by categor...


## 2. Customer category standardization

City labels are stripped and title-cased. Gender abbreviations and case variations map to `Male` or `Female`; blanks map to `Unknown`. Invalid or blank ages are replaced with the median valid age for the standardized city.

In [3]:
raw_city = raw["customers"]["city"].value_counts().rename_axis("raw_city").reset_index(name="count")
clean_city = clean["customers"]["city"].value_counts().sort_index().rename_axis("city").reset_index(name="count")
raw_gender = raw["customers"]["gender"].replace("", "<BLANK>").value_counts().rename_axis("raw_gender").reset_index(name="count")
clean_gender = clean["customers"]["gender"].value_counts().rename_axis("gender").reset_index(name="count")
print("Raw city labels")
display(raw_city)
print("Standardized cities")
display(clean_city)
print("Raw gender labels")
display(raw_gender)
print("Standardized genders")
display(clean_gender)
display(clean["customers"]["age"].describe().to_frame("clean_age"))

Raw city labels


,raw_city,count
0,Pune,148
1,Delhi,123
2,Hyderabad,119
3,Bengaluru,115
4,Mumbai,113
5,Chennai,106
6,chennai,44
7,hyderabad,40
8,delhi,36
9,pune,35


Standardized cities


,city,count
0,Bengaluru,197
1,Chennai,191
2,Delhi,210
3,Hyderabad,191
4,Mumbai,184
5,Pune,227


Raw gender labels


,raw_gender,count
0,female,136
1,FEMALE,122
2,F,120
3,m,118
4,f,117
5,Female,115
6,male,114
7,MALE,110
8,M,110
9,Male,108


Standardized genders


,gender,count
0,Female,610
1,Male,560
2,Unknown,30


,clean_age
count,1200.000000
mean,39.633333
std,12.046500
min,18.000000
25%,29.000000
50%,39.000000
75%,50.000000
max,60.000000


## 3. Duplicate and order-value resolution

The pipeline first removes exact duplicates, then keeps the latest timestamp for each order ID. Earlier `pending` states therefore resolve to their later `completed` or `cancelled` state. Invalid amounts are recalculated from product price × quantity × discount. Negative delivery durations become missing, and only completed orders receive category-median imputation.

In [4]:
order_checks = pd.DataFrame([
    {"check": "unique order IDs", "result": clean["orders"]["order_id"].is_unique},
    {"check": "pending rows", "result": int(clean["orders"]["status"].eq("pending").sum())},
    {"check": "missing amounts", "result": int(clean["orders"]["order_amount"].isna().sum())},
    {"check": "negative delivery durations", "result": int(clean["orders"]["delivery_minutes"].lt(0).sum())},
    {"check": "completed orders missing delivery", "result": int(clean["orders"].loc[clean["orders"]["status"].eq("completed"), "delivery_minutes"].isna().sum())},
])
display(order_checks)
display(clean["orders"].head())

,check,result
0,unique order IDs,True
1,pending rows,0
2,missing amounts,0
3,negative delivery durations,0
4,completed orders missing delivery,0


,order_id,customer_id,product_id,quantity,order_ts,status,discount_pct,order_amount,delivery_minutes
0,ORD-100001,C0001,P001,1,2025-09-16 08:07:21,completed,15,109.65,20
1,ORD-100002,C0001,P008,1,2025-09-03 07:34:01,completed,40,25.20,23
2,ORD-100003,C0001,P008,1,2025-04-08 17:08:14,completed,0,42.00,23
3,ORD-100004,C0001,P023,2,2025-07-29 22:27:32,completed,20,336.00,33
4,ORD-100005,C0002,P022,2,2025-07-18 15:52:20,completed,0,90.00,17


## 4. Cleaned CSV outputs

The trusted tables are saved under `data/processed/` for direct inspection.

In [5]:
processed_summary = pd.DataFrame([
    {"table": name, "rows": len(frame), "columns": len(frame.columns), "file": str(result["csv_paths"][name].relative_to(ROOT))}
    for name, frame in clean.items()
])
display(processed_summary)

,table,rows,columns,file
0,customers,1200,5,data/processed/customers.csv
1,products,40,4,data/processed/products.csv
2,orders,12309,9,data/processed/orders.csv
3,tickets,164,5,data/processed/tickets.csv


## 5. SQLite tables and relational checks

SQLite tables use primary keys, foreign keys, allowed-value checks, and numeric constraints. The database enables `PRAGMA foreign_keys` during loading.

In [6]:
db_path = result["db_path"]
with sqlite3.connect(db_path) as connection:
    tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", connection)
    status_summary = pd.read_sql_query("""
        SELECT status, COUNT(*) AS orders, ROUND(SUM(order_amount), 2) AS revenue_inr
        FROM orders GROUP BY status ORDER BY status
    """, connection)
    foreign_key_errors = connection.execute("PRAGMA foreign_key_check").fetchall()
display(tables)
display(status_summary)
print("Foreign-key violations:", len(foreign_key_errors))

,name
0,customers
1,orders
2,products
3,tickets


,status,orders,revenue_inr
0,cancelled,1199,333331.75
1,completed,11110,2998668.80


Foreign-key violations: 0


## 6. Pandas and SQL validation

The same metrics are calculated independently from the cleaned DataFrames and SQLite. Every row must match before the pipeline succeeds.

In [7]:
display(result["comparison"])
assert result["comparison"]["matches"].all()
print("ALL PANDAS/SQL CHECKPOINTS PASSED")

,metric,pandas_value,sql_value,matches
0,customers_rows,1200,1200,True
1,products_rows,40,40,True
2,orders_rows,12309,12309,True
3,tickets_rows,164,164,True
4,distinct_cities,6,6,True
5,completed_orders,11110,11110,True
6,cancelled_orders,1199,1199,True
7,completed_revenue_inr,2998668.80,2998668.80,True
8,tickets_without_order_id,8,8,True


ALL PANDAS/SQL CHECKPOINTS PASSED


## 7. Conclusions

- All 1,200 customers are retained with six standardized cities, three standardized gender values, and valid populated ages.
- The order history is reduced from 13,084 raw rows to 12,309 unique final orders; no pending state, nonnumeric amount, or negative delivery duration remains.
- All four cleaned tables are stored as CSV and in SQLite with referential integrity.
- Nine independently calculated Pandas and SQL metrics match.
- Empty and overlong ticket text remains intentionally available for privacy and model-guardrail work in a later part.